In [35]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [45]:
# read in state level external data
prev_year_state_margins = pd.read_csv("https://raw.githubusercontent.com/ecw70574/US-Census-Voting-and-Registration-DSCI-Capstone-Project/refs/heads/main/prev_year_state_margins.csv")
stateecon_indicators = pd.read_csv("https://raw.githubusercontent.com/ecw70574/US-Census-Voting-and-Registration-DSCI-Capstone-Project/refs/heads/main/stateecon_indicators.csv")


In [46]:
# read in and append the agg census data
import pandas as pd
import numpy as np
all_years = pd.DataFrame()
for i in range(10,24,2):
  # read in file
  filename = "panel_" + str(i) + ".csv"
  this_df = pd.read_csv("https://raw.githubusercontent.com/ecw70574/US-Census-Voting-and-Registration-DSCI-Capstone-Project/refs/heads/main/processed/" + filename)
  all_years = pd.concat([this_df, all_years])
all_years.head()


,states_encoded,age_group,income_group,education_group,year,weight,did_vote,sex_1,sex_2,marital_status_1,...,inflation_pct,unemployment_pct,approval_rating_pct,lag_popular_vote_pct,lag_electoral_votes,time_at_curr_address_-9,time_at_curr_address_-3,time_at_curr_address_-2,time_at_curr_address_5,time_at_curr_address_6
0,AK,18-24,high,associates,22,6166906.0,0.000000,0.000000,1.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,18-24,high,bachelors,22,7251923.0,0.000000,1.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AK,18-24,high,graduate,22,8726791.0,0.000000,0.000000,1.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AK,18-24,high,hs_grad,22,34631908.0,0.532165,0.410693,0.589307,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AK,18-24,high,less_hs,22,0.0,0.000000,0.000000,0.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# rename state column for consistency
all_years = all_years.rename(columns={"states_encoded": "state"})

In [49]:
all_years["year"].value_counts()

,count
year,
22,9000
20,9000
18,9000
16,9000
14,9000
12,9000
10,9000


In [69]:
# Merge all state/year level datasets
from sklearn.preprocessing import MinMaxScaler
merge1 = pd.merge(all_years, prev_year_state_margins, on=["year","state"], how='inner')
merge2 = pd.merge(merge1, stateecon_indicators, on=["year","state"], how='inner')



In [67]:
merge2.describe()

,year,weight,did_vote,sex_1,sex_2,marital_status_1,marital_status_2,marital_status_3,marital_status_4,marital_status_5,...,lag_electoral_votes,time_at_curr_address_-9,time_at_curr_address_-3,time_at_curr_address_-2,time_at_curr_address_5,time_at_curr_address_6,margin_of_victory_prev,oct_unemp,nov_unemp,percapita_personalincome
count,16920.000000,1.692000e+04,16920.000000,16920.000000,16920.000000,16920.000000,16920.000000,16920.000000,16920.000000,16920.000000,...,0.0,8460.000000,8460.000000,8460.000000,8460.000000,8460.000000,16920.000000,16920.000000,16920.000000,16920.000000
mean,16.000000,2.154156e+08,0.642591,0.413866,0.456051,0.419628,0.010109,0.047949,0.107557,0.016904,...,NaN,0.001178,0.006318,0.004950,0.108912,0.515602,0.170458,6.753191,6.598936,50586.989362
std,4.000118,3.575273e+08,0.369418,0.297481,0.306566,0.365577,0.055974,0.132585,0.190846,0.073080,...,NaN,0.018043,0.044386,0.039605,0.174413,0.361437,0.108426,1.805465,1.772987,9927.630976
min,12.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.002230,3.000000,3.000000,34812.000000
25%,12.000000,2.385477e+07,0.399016,0.181286,0.261035,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,0.000000,0.000000,0.000000,0.000000,0.167906,0.089527,5.300000,5.100000,43523.000000
50%,16.000000,8.519686e+07,0.768562,0.444009,0.486605,0.419158,0.000000,0.000000,0.000000,0.000000,...,NaN,0.000000,0.000000,0.000000,0.000000,0.560783,0.155384,6.900000,6.750000,50424.000000
75%,20.000000,2.504267e+08,1.000000,0.592531,0.645739,0.740120,0.000000,0.000000,0.154448,0.000000,...,NaN,0.000000,0.000000,0.000000,0.165024,0.828315,0.254448,8.100000,7.800000,56539.000000
max,20.000000,4.955198e+09,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,0.457695,11.300000,10.900000,77470.000000


In [52]:
data = merge2

In [53]:
import pandas as pd
import numpy as np
#from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.sequence import pad_sequences

# revised methodology: avoids leakage from same group over time

# 1. load data
# reading in resulting CSV from aggregate_and_build.py script
# data = pd.read_csv("lstm_preprocessed_data.csv")
# data = pd.read_csv("processed/all_years_aggregated.csv")
# data = pd.read_csv("all_years_aggregated.csv")

In [66]:

# 2. group id
data["group_id"] = (
    data["state"].astype(str) + "_" +
    data["education_group"].astype(str) + "_" +
    data["age_group"].astype(str) + "_" +
    data["income_group"].astype(str)
)

# adding lag and interaction terms

# 3. survival filter
group_survival = (
    data.groupby("group_id")["year"]
    .nunique()
    .reset_index(name='year_count')
)

group_survival.sort_values(by='year_count', ascending=False).head(30)


,group_id,year_count
8459,WY_some_college_65+_upper_middle,2
0,AK_associates_18-24_high,2
1,AK_associates_18-24_low,2
2,AK_associates_18-24_lower_middle,2
3,AK_associates_18-24_middle,2
4,AK_associates_18-24_upper_middle,2
5,AK_associates_25-34_high,2
6,AK_associates_25-34_low,2
7,AK_associates_25-34_lower_middle,2
8,AK_associates_25-34_middle,2


In [60]:
# making sure groups are consistent between all 8 years
# true time series will track same groups over time
surviving_groups = group_survival[group_survival["year_count"] >= 8]["group_id"]

model_data = data[data["group_id"].isin(surviving_groups)].copy()

# 4. sort
model_data = model_data.sort_values(["group_id", "year"])

# 5. fill missing
model_data = model_data.fillna(0)

In [61]:
model_data.head()

,state,age_group,income_group,education_group,year,weight,did_vote,sex_1,sex_2,marital_status_1,...,time_at_curr_address_-9,time_at_curr_address_-3,time_at_curr_address_-2,time_at_curr_address_5,time_at_curr_address_6,margin_of_victory_prev,oct_unemp,nov_unemp,percapita_personalincome,group_id


In [ ]:
# remove columns we already grouped on - don't need them as features! - redundant
# also werent properly encoded so were causing typecast errors
model_data = model_data.drop(columns=["age_group", "education_grouped","family_income_grouped"])

# 6. features
# everything except the group id, year, and target variable
feature_cols = [
    c for c in model_data.columns
    if c not in ["group_id", "year", "did_vote_1"]
]
# 7. buidling sequences for each group
# need to filter for years then build sequences after
train_years = 2018
val_years   = 2022
test_years  = 2024
def build_sequences(df, max_year):
    X_seq, y_seq = [], []

    for gid, g in df.groupby("group_id"):
        g = g.sort_values("year")

        g = g[g["year"] <= max_year]   # filter to find rows in corrct year

        if len(g) < 2:
            continue

        X_seq.append(g[feature_cols].values) # only rows with correct year added to sequence
        y_seq.append(g["did_vote_1"].values[-1]) # only rows with coreect year added to sequence

    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


X_train, y_train = build_sequences(model_data, 2018)
X_val, y_val     = build_sequences(model_data, 2022)
X_test, y_test   = build_sequences(model_data, 2024)

# Normalize 0-1 for all features
scaler = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)
X_test = scalar.fit_transform(X_test)

'''
# THIS IS INVALID BECAUSE HAS LEAKAGE
# split into training and testing
# this is year level split
# each year snapshot would either be in train, test, or validation set
# train on years <- 2018, validate on 2020-2022, test on 2024
train_data = model_data[model_data["year"] <= 18] # for training
val_data   = model_data[(model_data["year"] > 18) & (model_data["year"] <= 22)] # for tuning/selection
# after final model has been tuned/optimized, avoid data leakage
test_data  = model_data[model_data["year"] == 24] # for testing performance on unseen observations
'''

'\n# THIS IS INVALID BECAUSE HAS LEAKAGE\n# split into training and testing \n# this is year level split\n# each year snapshot would either be in train, test, or validation set\n# train on years <- 2018, validate on 2020-2022, test on 2024\ntrain_data = model_data[model_data["year"] <= 18] # for training\nval_data   = model_data[(model_data["year"] > 18) & (model_data["year"] <= 22)] # for tuning/selection\n# after final model has been tuned/optimized, avoid data leakage\ntest_data  = model_data[model_data["year"] == 24] # for testing performance on unseen observations \n'

In [ ]:
print(X_train.shape)


(1620, 8, 59)


In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

# Define input parameters
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]
# set up the architecture - subject to change
model = Sequential([
    # Input layer with input_shape matching your sequences
    LSTM(64, input_shape=(n_timesteps, n_features), return_sequences=False),

    # Optional: Add Dropout to prevent overfitting
    Dropout(0.2),

    # Output layer: 1 for single-value prediction, or len(y_train[0]) for multi-step
    Dense(1, activation='sigmoid')  # Sigmoid to bound 0-1
])

model.compile(optimizer='adam', loss='mse', metrics=["mae"])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
type(X_train)

numpy.ndarray

In [ ]:
type(y_train)

numpy.ndarray

In [ ]:
X_train.shape

(1620, 8, 59)

In [ ]:
X_train.shape[1] == 8

True

In [ ]:
print(X_train.dtype)
print(type(X_train[0][0][0]))

float32
<class 'numpy.float32'>


In [ ]:
print(X_train.dtype)
print(type(X_train[0]))

float32
<class 'numpy.ndarray'>


In [ ]:
# Train the model
# history = model.fit(X_train, y_train, epochs=20, batch_size=256, verbose=1)

In [ ]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,           # Wait 5 epochs for improvement
    restore_best_weights=True # Revert to the best model
)

# Use in model.fit
model.fit(
    X_train, y_train,
    epochs=100,           # Set a high max, let early stopping take over
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose = 1
)


Epoch 1/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0931 - mae: 0.2632 - val_loss: 0.0698 - val_mae: 0.2275
Epoch 2/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0682 - mae: 0.2207 - val_loss: 0.0585 - val_mae: 0.2026
Epoch 3/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0617 - mae: 0.2048 - val_loss: 0.0569 - val_mae: 0.1961
Epoch 4/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0603 - mae: 0.1996 - val_loss: 0.0567 - val_mae: 0.1941
Epoch 5/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0609 - mae: 0.1990 - val_loss: 0.0567 - val_mae: 0.1938
Epoch 6/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0610 - mae: 0.1984 - val_loss: 0.0566 - val_mae: 0.1928
Epoch 7/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0614 - mae: 0.1995 - val_loss: 0.0566 - val_mae: 0.1925
Epoch 8/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0595 - mae: 0.1947 - val_loss: 0.0565 - val_mae: 0.1925
Epoch 9/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.

In [ ]:
val_loss, val_mae = model.evaluate(X_val, y_val)
print("Validation Loss:", val_loss)
print("Validation MAE:", val_mae)

51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0543 - mae: 0.1895
Validation Loss: 0.05434552952647209
Validation MAE: 0.1894661784172058


In [ ]:
model.metrics_names

['loss', 'compile_metrics']

In [ ]:
y_pred = model.predict(X_val)

51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [ ]:
y_pred.shape

(1620, 1)

In [ ]:
y_val.shape

(1620,)

In [ ]:
y_pred = y_pred.flatten()
y_true = y_val.flatten()

In [ ]:
y_true.shape

(1620,)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

MAE: 0.18946616351604462
MSE: 0.05434553325176239
RMSE: 0.23312128442457242


In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_true, y_pred)
print("R²:", r2)

R²: 0.1780916452407837


In [ ]:
y_train

array([0.8633987 , 0.        , 1.        , ..., 0.88299507, 0.8647464 ,
       0.9059556 ], dtype=float32)